In [319]:
import geopandas as gpd 
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium import plugins

In [320]:
#add the fire data from 2023
fire_2023 = pd.read_csv("../data/processed/fire_2023_clean.csv")

#Now we create point geometries
fire_2023_gdf = gpd.GeoDataFrame(
    fire_2023, geometry=gpd.points_from_xy(
        fire_2023.longitude, fire_2023.latitude),
    crs="EPSG:4326")

fire_2023_gdf.to_file("../data/processed/fire_2023_points.geojson", driver="GeoJSON")

In [321]:
#First we create a new Folium map, which is centered to Canada. 
canada_map = folium.Map(
    location = [57.03,-98.82],
    zoom_start =4,
    tiles = "CartoDB.DarkMatterNoLabels", 
)

#Then we display the cities on to canada
#add the city data

selected_cities = pd.read_csv("../data/processed/selected_cities_canada.csv")

#Now we create point geometries
selected_cities_gdf = gpd.GeoDataFrame(
    selected_cities, geometry=gpd.points_from_xy(
        selected_cities.lng, selected_cities.lat),
    crs="EPSG:4326")

selected_cities_gdf.to_file("../data/processed/selected_cities_canada.geojson", driver="GeoJSON")

folium.GeoJson(
    selected_cities_gdf,
    name = "Cities of Canada",
    tooltip=folium.GeoJsonTooltip(
        fields=["city","population","country"], aliases=["Name:","Population:","Country:"]),
).add_to(canada_map)

#Heatmap for weighted fires

heat_data = [[point.xy[1][0], point.xy[0][0], weight] 
             for point, weight in zip (
                 fire_2023_gdf.geometry,
                 fire_2023_gdf["class_weight"])
                 ]

plugins.HeatMap(
    heat_data,
    radius=15,
    blur=5,
    min_opacity=0.25,
    name = "Wildfires in 2023"
).add_to(canada_map)

In [322]:
import pandas as pd
import geopandas as gpd 
import requests
import folium

map_key = "b73850f35cfea7f219a3969b6df1d9a6"
api_url = f"https://firms.modaps.eosdis.nasa.gov/usfs/api/area/csv/{map_key}/VIIRS_NOAA20_NRT/world/5/2026-05-07"

response = requests.get(api_url)
response.status_code


200

In [323]:
if response.status_code == 200:
    fire_live_df = pd.read_csv(api_url)
    fire_live_df.to_csv("../data/processed/fire_live.csv", index=False)
    #to not load the whole world, we define a bounding box
    north = 73.226700
    south = 41.640078
    west  = -140.625000
    east  = -49.218750
    fire_live_df = fire_live_df[
        (fire_live_df["latitude"] >= south) &
        (fire_live_df["latitude"] <= north) &
        (fire_live_df["longitude"] >= west) &
        (fire_live_df["longitude"] <= east)
        ]
    print("success")
else:
    print("Error")

success


In [324]:
live_heatmap = [
    [row["latitude"], row["longitude"]]
    for _, row in fire_live_df.iterrows()
] 
plugins.HeatMap(
    live_heatmap,
    radius=15,
    blur=5,
    min_opacity=0.25,
    name = "Live wildfire",
    show = False
).add_to(canada_map)

folium.LayerControl(collapsed=False).add_to(canada_map)

In [325]:
canada_map